In [64]:
import importlib.util
import sys, sysconfig
from pathlib import Path
libdnf5_name = "libdnf5"
libdnf5_path = Path(
    f"{sysconfig.get_paths()["stdlib"]}/site-packages/libdnf5/__init__.py")
if not libdnf5_path.exists():
    raise RuntimeError("libdnf5 is not installed")

spec = importlib.util.spec_from_file_location(libdnf5_name, libdnf5_path)
if spec is None:
    raise RuntimeError(f"Unable to create spec for {libdnf5_name}, {libdnf5_path}")
libdnf5 = importlib.util.module_from_spec(spec)
sys.modules[libdnf5_name] = libdnf5
if spec.loader is None:
    raise RuntimeError(f"Unable to create spec for {libdnf5_name}, {libdnf5_path}")
spec.loader.exec_module(libdnf5)

pkg_dict = dict()

base = libdnf5.base.Base()
base.load_config()
base.setup()
repo_sack = base.get_repo_sack()
repo_sack.create_repos_from_dir("/etc/yum.repos.d/")
repo_sack.load_repos(libdnf5.base.repo.Repo.Type_AVAILABLE)

package_name = "ps"

# query = libdnf5.rpm.PackageQuery(base)
# query.filter_name(package_name, libdnf5.common.QueryCmp_CONTAINS)
# query.filter_arch('x86_64')
# for q in query:
#     print(q.get_nevra())
# print("--------------------------------------------------")
query = libdnf5.rpm.PackageQuery(base)
query.filter_name(package_name)
query.filter_arch('x86_64')
for q in query:
    print(q.get_nevra())    

In [4]:
import networkx as nx
g = nx.Graph()
g.add_nodes_from(pkg_dict.values())
for pkg in pkg_dict.values():
    reldeps = pkg.get_depends()
    for reldep in reldeps:
        dep_name = reldep.get_name()
        print(dep_name)
        if dep_name in pkg_dict:
            g.add_edge(pkg, pkg_dict[dep_name])



/usr/bin/sh
perl(strict)
perl(warnings)
/usr/bin/perl
perl(File::Basename)
perl(File::Spec)
perl(Getopt::Long)
perl(lib)
perl(IPC::Open2)
perl(Git)
perl(Term::ReadKey)
perl(:VERSION)
git-core
perl-Git
git-core-doc
rtld(GNU_HASH)
libc.so.6(GLIBC_2.38)(64bit)
libtinfo.so.6()(64bit)
filesystem
perl(strict)
perl-libs
perl(warnings)
perl(Exporter)
perl(base)
perl(constant)
perl(Getopt::Long)
perl(overload)
perl(Text::ParseWords)
perl(:VERSION)
perl(Pod::Usage)
rtld(GNU_HASH)
libc.so.6(GLIBC_2.38)(64bit)
perl(strict)
perl-libs
perl(Carp)
perl(Exporter)
perl(constant)
perl(File::Spec)
perl(Scalar::Util)
libperl.so.5.40()(64bit)
perl(Cwd)
perl(Errno)
perl(:MODULE_COMPAT_5.40.1)
perl(File::Spec::Unix)
perl(XSLoader)
rtld(GNU_HASH)
perl(strict)
perl-libs
perl(warnings)
perl(Exporter)
perl(vars)
libperl.so.5.40()(64bit)
perl(:MODULE_COMPAT_5.40.0)
perl(DynaLoader)
libc.so.6(GLIBC_2.28)(64bit)
rtld(GNU_HASH)
libc.so.6(GLIBC_2.38)(64bit)
/usr/bin/sh
libz.so.1()(64bit)
libcrypto.so.3()(64bit)
libcry

In [11]:
from pfparsons.oci import DNF

dnf = DNF()
d = dnf.dependencies("graphviz")
if d is not None:
    s = d.total_install_size
    print(s / 1024 / 1024)

158.71027755737305


In [1]:
dnf.dependencies("python3.12-3.12.12-1.fc42.x86_64")

NameError: name 'dnf' is not defined

In [1]:
from pfparsons.oci import DNF

dnf = DNF()
d = dnf.dependencies("git")
if d is not None:
    s = d.total_install_size
    print(s / 1024 / 1024)

tx: <libdnf5.base.Transaction; proxy of <Swig Object of type 'libdnf5::base::Transaction *' at 0x7f7cd078aee0> >


LookupError: Unable to find package named git 

In [1]:
from pfparsons.oci import DNF
from pathlib import Path

dnf = DNF()
basedir = Path('/workspaces/pfparsons-examples/infra/oci/images/fedora')
packages = []
for path in basedir.glob("**/rpm-list-*.txt"):
    if path.is_file():
        text = path.read_text()
        for line in text.split("\n"):
            clean_line = line.strip()
            if not clean_line.startswith("#") and len(clean_line) > 0:
                packages.append(clean_line)
for p in packages:
    print(f"getting deps for [{p}]")
    dnf.dependencies(p)


getting deps for [python3.12]
getting deps for [which]
getting deps for [procps-ng]
getting deps for [wget2]
getting deps for [curl]
getting deps for [zip]
getting deps for [unzip]
getting deps for [tar]
getting deps for [golang]
getting deps for [gcc]
getting deps for [gcc-c++]
getting deps for [make]
getting deps for [ninja-build]
getting deps for [cmake]
getting deps for [git]
getting deps for [patch]
getting deps for [valgrind]
getting deps for [gdb]
getting deps for [clang]
getting deps for [polly]
getting deps for [lld]
getting deps for [clang-libs]
getting deps for [clang-tools-extra]
getting deps for [clang-tools-extra-devel]
getting deps for [clang-analyzer]
getting deps for [llvm]
getting deps for [lldb]
getting deps for [libdnf5-devel]
getting deps for [rpm-build]
getting deps for [rpmdevtools]
getting deps for [python3-devel]


In [2]:
from pfparsons.oci import PackageNode, _make_base
def sort_by_size(s: list[PackageNode]): return reversed(sorted(s, key=lambda x: x.total_install_size))

def print_deps(s: list[PackageNode]):
    for n in sort_by_size(s):
        label = f"{n.name} ".ljust(32,'—')
        size_mb = str(round(n.total_install_size / 1024 / 1024,1)).rjust(5,' ')
        print(f"{label} {size_mb } MB")

def explain_size(self, spec: str):
    r = [k for k in self.nodes.keys() if k.startswith(spec)]
    if len(r) > 0:
        node = self.nodes[r[0]]
        size_mb = str(round(node.total_install_size / 1024 / 1024,1)).rjust(8,' ')
        print(f"Total Size: {size_mb} MB")
        size_mb = str(round(node.install_size / 1024 / 1024,1)).rjust(8,' ')
        print(f"{spec}: {size_mb}")
        print_deps(node.children)

def reldeps(self, spec: str):
    base = _make_base()
    goal = libdnf5.base.Goal(self.base)
    goal.add_install(spec)
    tx = goal.resolve()
    pkg_tx_list = tx.get_transaction_packages()
    pkg_tx = pkg_tx_list[0]
    pkg = pkg_tx.get_package()
    print(pkg)
    for d in pkg.get_depends():
        print(d.get_name())

#print_deps(dnf.nodes.values())
reldeps(dnf, 'cpp')

NameError: name 'libdnf5' is not defined

In [4]:
dnf.dependencies("rtld(GNU_HASH)").children

[tzdata-0:2025b-1.fc42.noarch,
 glibc-gconv-extra-0:2.41-11.fc42.x86_64,
 fedora-release-identity-basic-0:42-30.noarch,
 fedora-gpg-keys-0:42-1.noarch,
 fedora-repos-0:42-1.noarch,
 fedora-release-common-0:42-30.noarch,
 fedora-release-0:42-30.noarch,
 glibc-minimal-langpack-0:2.41-11.fc42.x86_64,
 libgcc-0:15.2.1-3.fc42.x86_64,
 filesystem-0:3.18-47.fc42.x86_64,
 ncurses-base-0:6.5-5.20250125.fc42.noarch,
 ncurses-libs-0:6.5-5.20250125.fc42.x86_64,
 bash-0:5.2.37-1.fc42.x86_64,
 setup-0:2.15.0-13.fc42.noarch,
 glibc-common-0:2.41-11.fc42.x86_64,
 basesystem-0:11-22.fc42.noarch,
 glibc-0:2.41-11.fc42.x86_64]

In [59]:
explain_size(dnf,'clang-devel')

Total Size:   1334.8 MB
clang-devel:     28.4
llvm-static ———————————————————— 365.1 MB
llvm-libs —————————————————————— 137.1 MB
clang-libs ————————————————————— 120.2 MB
gcc ———————————————————————————— 111.2 MB
llvm ———————————————————————————  89.7 MB
clang-tools-extra ——————————————  67.2 MB
gcc-c++ ————————————————————————  41.4 MB
compiler-rt ————————————————————  40.4 MB
python3-libs ———————————————————  40.1 MB
cpp ————————————————————————————  37.9 MB
llvm-devel —————————————————————  28.9 MB
binutils ———————————————————————  25.8 MB
libstdc++-devel ————————————————  16.1 MB
systemd-udev ———————————————————  12.2 MB
systemd ————————————————————————  12.1 MB
coreutils-common ———————————————  11.1 MB
cracklib-dicts —————————————————   9.3 MB
bash ———————————————————————————   8.2 MB
openssl-libs ———————————————————   7.8 MB
glibc-gconv-extra ——————————————   7.2 MB
kernel-headers —————————————————   6.7 MB
glibc ——————————————————————————   6.6 MB
xkeyboard-config —————————————

In [ ]:
lookup_deps(dnf,'clang-devel')


llvm-static ———————————————————— 365.1 MB
llvm-libs —————————————————————— 137.1 MB
clang-libs ————————————————————— 120.2 MB
gcc ———————————————————————————— 111.2 MB
llvm ———————————————————————————  89.7 MB
clang-tools-extra ——————————————  67.2 MB
gcc-c++ ————————————————————————  41.4 MB
compiler-rt ————————————————————  40.4 MB
python3-libs ———————————————————  40.1 MB
cpp ————————————————————————————  37.9 MB
llvm-devel —————————————————————  28.9 MB
binutils ———————————————————————  25.8 MB
libstdc++-devel ————————————————  16.1 MB
systemd-udev ———————————————————  12.2 MB
systemd ————————————————————————  12.1 MB
coreutils-common ———————————————  11.1 MB
cracklib-dicts —————————————————   9.3 MB
bash ———————————————————————————   8.2 MB
openssl-libs ———————————————————   7.8 MB
glibc-gconv-extra ——————————————   7.2 MB
kernel-headers —————————————————   6.7 MB
glibc ——————————————————————————   6.6 MB
xkeyboard-config ———————————————   6.6 MB
coreutils ——————————————————————  

In [72]:
from pfparsons.oci import _make_base
def _resolve_spec_settings():
    settings = libdnf5.base.GoalJobSettings()
    #settings.set_group_with_name(True)
    settings.set_with_binaries(True)
    #settings.set_with_provides(True)
    settings.with_binaries = True
    return settings

def resolve_spec(spec: str):
    base = _make_base()
    settings = _resolve_spec_settings()
    match, spec_nevra = libdnf5.rpm.PackageQuery(base).resolve_pkg_spec(spec, settings, False)
    #if not match: # or spec_nevra.has_just_name():
    #    return False
    print(list(query))

def provides(spec: str):
    base = libdnf5.base.Base()
    base.load_config()
    base.setup()
    repo_sack = base.get_repo_sack()
    repo_sack.create_repos_from_dir("/etc/yum.repos.d/")
    repo_sack.load_repos(libdnf5.base.repo.Repo.Type_AVAILABLE)
    query = libdnf5.rpm.PackageQuery(base)
    query.filter_provides(spec)
    results = []
    for q in query:
        results.append(q.get_nevra())
    if len(results) > 0:
        return results
    return results
    
    # packages = [p.get_evr() for p in query]
    # libdnf5.rpm.rpmvercmp(packages[1], packages[0]))

#print(provides("libc.so.6()(64bit)"))
#print(provides("/bin/ps"))
#resolve_spec('procps-ng')
resolve_spec("rtld(GNU_HASH)")

resolve_spec("rtld")


[]
[]
